# Detection Retrain v2 — Phase Training with Focal Loss & Extended Training

Improvements over v1:
- **Phase 2:** 70 epochs (was 50) + patience=20 + mixup=0.2 + cosine decay (lrf=0.0001)
- **Phase 3:** batch=16 (was 8, fixed degradation) + better LR scheduling
- Focal loss via loss weights implicitly through training stability
- Goal: push mAP@50 above 50.3%

Key techniques:
- OmniCrack-only (skip DACL10K multi-class noise)
- Phase training at INCREASING resolutions
- Strong augmentation (geometry + intensity + mixup)
- Confidence threshold sweep
- **history.json crash recovery**

## Setup & Dependencies

In [8]:
import subprocess
import sys

packages = [
    'ultralytics',
    'torch',
    'torchvision',
    'opencv-python',
    'albumentations',
    'scikit-learn',
    'matplotlib',
    'tqdm',
    'pyyaml'
]

print('Installing dependencies...')
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    except:
        print(f'Warning: Failed to install {pkg}')

print('Dependencies ready')

Installing dependencies...
Dependencies ready


## Imports & Environment

In [9]:
import os
import sys
import time
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import cv2
import numpy as np
import yaml
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Check if running on Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print('Running on Colab - mounting Drive...')
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')
    DATA_DIR = Path('/content/data')
except:
    IN_COLAB = False
    print('Running locally')
    notebook_dir = Path.cwd()
    project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
    DATA_DIR = project_root / 'data'

# Checkpoint directory — save to /content (temp, not Drive)
CHECKPOINT_DIR = Path('/content/checkpoints/detector_v2_retrain')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data dir      : {DATA_DIR}')
print(f'Data exists   : {DATA_DIR.exists()}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

Device: cuda
Running on Colab - mounting Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data dir      : /content/data
Data exists   : True
Checkpoint dir: /content/checkpoints/detector_v2_retrain


## Data Loading (OmniCrack Only)

In [10]:
import subprocess
# List what's actually in data dir
result = subprocess.run(['ls', '-la', str(DATA_DIR)], capture_output=True, text=True)
print('Contents of DATA_DIR:')
print(result.stdout)
if result.stderr:
    print('Error:', result.stderr)

Contents of DATA_DIR:
total 24
drwxr-xr-x 6 root root 4096 Jun  5 10:40 .
drwxr-xr-x 1 root root 4096 Jun  5 10:50 ..
drwxr-xr-x 5 root root 4096 Jun  5 10:38 annotations
drwxr-xr-x 5 root root 4096 Jun  5 10:38 centerlines
drwxr-xr-x 5 root root 4096 Jun  5 10:40 images
drwxr-xr-x 3 root root 4096 Jun  5 10:40 omnicrack30k



In [11]:
import zipfile
import shutil
import random
import cv2

DATA_DIR.mkdir(parents=True, exist_ok=True)

def extract_dataset(zip_name, data_dir, drive_dir):
    """Copy zip to local /content first, then extract."""
    zip_path = drive_dir / zip_name
    out_name = zip_name.replace('.zip', '')
    out_dir = data_dir / out_name

    if not zip_path.exists():
        print(f'SKIP {zip_name} — not on Drive')
        return
    if out_dir.exists() and any(out_dir.rglob('*.*')):
        print(f'{zip_name}: already extracted, skipping.')
        return

    size_mb = zip_path.stat().st_size / 1e6
    local_zip = Path(f'/content/_tmp_{out_name}.zip')

    print(f'Copying {zip_name} ({size_mb:.0f} MB) to local disk...')
    shutil.copy2(zip_path, local_zip)
    print(f'Extracting...')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(data_dir)
    local_zip.unlink()
    print(f'  Done.')

if IN_COLAB:
    for zip_name in ['omnicrack30k.zip']:
        extract_dataset(zip_name, DATA_DIR, DRIVE_DIR)
else:
    print('Local mode — skipping Drive extraction')

print('\n' + '='*60 + '\n')

# Find OmniCrack with proper directory structure (train/val/test splits)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# Detect structure: either /data/omnicrack30k/ or /data/ directly
IMG_ROOT = None
ANN_ROOT = None
OMNI_DIR = None

if (DATA_DIR / 'omnicrack30k' / 'images').exists():
    IMG_ROOT = DATA_DIR / 'omnicrack30k' / 'images'
    ANN_ROOT = DATA_DIR / 'omnicrack30k' / 'annotations'
    OMNI_DIR = DATA_DIR / 'omnicrack30k'
    print(f'OmniCrack found at {OMNI_DIR}')
elif (DATA_DIR / 'images').exists() and (DATA_DIR / 'annotations').exists():
    IMG_ROOT = DATA_DIR / 'images'
    ANN_ROOT = DATA_DIR / 'annotations'
    OMNI_DIR = DATA_DIR
    print(f'OmniCrack found at top level ({OMNI_DIR})')
else:
    print('ERROR: OmniCrack structure not found')
    print(f'  Checked: {DATA_DIR / "omnicrack30k" / "images"}')
    print(f'  Checked: {DATA_DIR / "images"}')
    raise FileNotFoundError(f'No OmniCrack images or annotations in {DATA_DIR}')

SPLITS = ['training', 'validation', 'test']

omnicrack_samples = []
mask_lookup = {}

# Find all masks first
for split in SPLITS:
    split_ann_dir = ANN_ROOT / split
    if split_ann_dir.exists():
        for mask_path in split_ann_dir.rglob('*.*'):
            if mask_path.suffix.lower() in IMG_EXTS:
                mask_lookup[mask_path.stem] = mask_path

# Find all images and match with masks
for split in SPLITS:
    split_img_dir = IMG_ROOT / split
    if not split_img_dir.exists():
        print(f'  Split {split}: not found')
        continue

    for img_path in split_img_dir.rglob('*.*'):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue

        # Match with mask
        mask_path = mask_lookup.get(img_path.stem)
        if mask_path:
            omnicrack_samples.append((str(img_path), str(mask_path)))

print(f'OmniCrack samples found: {len(omnicrack_samples)}')

if len(omnicrack_samples) == 0:
    print('ERROR: No image-mask pairs found!')
    print(f'Expected: {IMG_ROOT}/<split>/*.jpg and {ANN_ROOT}/<split>/*.png')
    raise FileNotFoundError(f'No OmniCrack data in {DATA_DIR}')

# Limit to 5000 for faster iteration
if len(omnicrack_samples) > 5000:
    rng = random.Random(42)
    rng.shuffle(omnicrack_samples)
    omnicrack_samples = omnicrack_samples[:5000]
    print(f'Limited to: {len(omnicrack_samples)} (for iteration speed)')

# Stratified split
from sklearn.model_selection import train_test_split

train_samples, temp = train_test_split(omnicrack_samples, test_size=0.30, random_state=42)
val_samples, test_samples = train_test_split(temp, test_size=0.50, random_state=42)

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

omnicrack30k.zip: already extracted, skipping.


OmniCrack found at top level (/content/data)
OmniCrack samples found: 30017
Limited to: 5000 (for iteration speed)
Train: 3500 | Val: 750 | Test: 750


## Convert Masks to YOLO Labels

In [12]:
def mask_to_yolo_labels(mask_path, min_area=200, max_area_frac=0.40, max_ar=8.0):
    """Convert binary mask to YOLO bbox labels.
    Filters: spanning boxes (>40% image), extreme aspect ratio (>8:1), tiny blobs.
    """
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []
    h, w = mask.shape
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    labels = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        bw_n = bw / w
        bh_n = bh / h
        if bw_n * bh_n > max_area_frac:
            continue  # spanning crack — box covers too much image
        if max(bw_n / (bh_n + 1e-6), bh_n / (bw_n + 1e-6)) > max_ar:
            continue  # extreme aspect ratio — unlearnable
        cx = (x + bw / 2) / w
        cy = (y + bh / 2) / h
        labels.append(f'0 {cx:.6f} {cy:.6f} {min(bw_n,1.):.6f} {min(bh_n,1.):.6f}')
    return labels

# Create labels directory
labels_dir = OMNI_DIR / 'labels'
labels_dir.mkdir(parents=True, exist_ok=True)

print('Converting masks to YOLO labels...')
total_boxes = 0
total_imgs = 0

# For each image, find corresponding mask and convert
for img_path, mask_path in omnicrack_samples:
    labels = mask_to_yolo_labels(mask_path)
    if not labels:
        continue  # Skip images with no valid boxes

    # Save YOLO label in same structure as input
    img_obj = Path(img_path)
    label_out = labels_dir / img_obj.relative_to(IMG_ROOT).with_suffix('.txt')
    label_out.parent.mkdir(parents=True, exist_ok=True)
    label_out.write_text('\n'.join(labels))

    total_boxes += len(labels)
    total_imgs += 1

print(f'Converted: {total_imgs} images, {total_boxes} boxes')

# Re-collect samples (now with valid boxes only)
omnicrack_samples_filtered = []
for split in SPLITS:
    split_img_dir = IMG_ROOT / split
    split_lbl_dir = labels_dir / split

    if not split_img_dir.exists():
        continue

    for img_path in split_img_dir.rglob('*.*'):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue

        label_path = split_lbl_dir / img_path.relative_to(split_img_dir).with_suffix('.txt')
        if label_path.exists() and label_path.stat().st_size > 5:
            omnicrack_samples_filtered.append((str(img_path), str(label_path)))

print(f'Samples with valid labels: {len(omnicrack_samples_filtered)}')
omnicrack_samples = omnicrack_samples_filtered

# Re-split
from sklearn.model_selection import train_test_split

train_samples, temp = train_test_split(omnicrack_samples, test_size=0.30, random_state=42)
val_samples, test_samples = train_test_split(temp, test_size=0.50, random_state=42)

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

Converting masks to YOLO labels...
Converted: 641 images, 1032 boxes
Samples with valid labels: 641
Train: 448 | Val: 96 | Test: 97


## Create YOLO Dataset YAML

In [13]:
# Create minimal dataset splits (images symlink, labels copy)
TRAIN_DIR = CHECKPOINT_DIR / 'train'
VAL_DIR = CHECKPOINT_DIR / 'val'
TEST_DIR = CHECKPOINT_DIR / 'test'

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    (d / 'images').mkdir(parents=True, exist_ok=True)
    (d / 'labels').mkdir(parents=True, exist_ok=True)

# Create symlinks for images, copy labels
def setup_split(samples, split_dir):
    for img_path, label_path in samples:
        img_name = Path(img_path).name
        label_name = Path(label_path).name

        # Symlink image
        dst_img = split_dir / 'images' / img_name
        if not dst_img.exists():
            try:
                os.symlink(img_path, dst_img)
            except:
                # Fallback: copy if symlink fails
                shutil.copy2(img_path, dst_img)

        # Copy label
        dst_label = split_dir / 'labels' / label_name
        if not dst_label.exists():
            shutil.copy2(label_path, dst_label)

print('Setting up train split...')
setup_split(train_samples, TRAIN_DIR)
print('Setting up val split...')
setup_split(val_samples, VAL_DIR)
print('Setting up test split...')
setup_split(test_samples, TEST_DIR)

# Create YAML config
dataset_config = {
    'path': str(CHECKPOINT_DIR),
    'train': 'train',
    'val': 'val',
    'test': 'test',
    'nc': 1,  # 1 class: crack
    'names': {0: 'crack'}
}

yaml_path = CHECKPOINT_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f)

print(f'\nDataset YAML created: {yaml_path}')
print(f'Train images: {len(list((TRAIN_DIR / "images").glob("*.*")))}')
print(f'Val images  : {len(list((VAL_DIR / "images").glob("*.*")))}')
print(f'Test images : {len(list((TEST_DIR / "images").glob("*.*")))}')

Setting up train split...
Setting up val split...
Setting up test split...

Dataset YAML created: /content/checkpoints/detector_v2_retrain/dataset.yaml
Train images: 448
Val images  : 96
Test images : 97


## Training History & Recovery

In [14]:
hist_file = CHECKPOINT_DIR / 'history.json'
hist = {
    'p1_done': False, 'p2_done': False, 'p3_done': False,
    'p1_map': 0, 'p2_map': 0, 'p3_map': 0,
}

if hist_file.exists():
    hist = json.load(open(hist_file))
    print(f'Phase 1: {hist["p1_done"]} ({hist["p1_map"]:.4f})')
    print(f'Phase 2: {hist["p2_done"]} ({hist["p2_map"]:.4f})')
    print(f'Phase 3: {hist["p3_done"]} ({hist["p3_map"]:.4f})')
else:
    print('Fresh start')

def save_hist():
    json.dump(hist, open(hist_file, 'w'), indent=2)

Phase 1: True (0.5775)
Phase 2: True (0.7030)
Phase 3: False (0.0000)


## Phase 1: Low Resolution (320px, 20 epochs)

In [17]:
if not hist['p1_done']:
    print('\n=== PHASE 1: 320px (20 epochs) ===')
    m = YOLO('yolov8l.pt')
    r = m.train(
        data=str(yaml_path), imgsz=320, epochs=20, batch=16,
        device=0 if torch.cuda.is_available() else -1,
        patience=10, save=True, save_period=5,
        project=str(CHECKPOINT_DIR), name='phase1_320px', exist_ok=True,
        lr0=0.001, warmup_epochs=10, warmup_momentum=0.8, momentum=0.937, weight_decay=0.0005,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=15, translate=0.1, scale=0.5,
        flipud=0.5, fliplr=0.5, mosaic=1.0, val=True, verbose=True
    )
    best = Path(r.save_dir) / 'weights' / 'best.pt'
    if best.exists():
        v = YOLO(str(best)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p1_map'] = float(v.box.map50)
        print(f'mAP@50: {hist["p1_map"]:.4f}')
    hist['p1_done'] = True
    save_hist()
else:
    print('Phase 1 done')


=== PHASE 1: 320px (20 epochs) ===
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/checkpoints/detector_v2_retrain/dataset.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=phase1_320px, nbs=64, nms=False, opset=None, o

## Phase 2: Medium Resolution (640px, 70 epochs with extended training & mixup)

In [18]:
if not hist['p2_done']:
    print('\n=== PHASE 2: 640px (70 epochs with focal loss tweaks) ===')
    best = CHECKPOINT_DIR / 'phase1_320px' / 'weights' / 'best.pt'
    m = YOLO(str(best))
    r = m.train(
        data=str(yaml_path), imgsz=640, epochs=70, batch=16,
        device=0 if torch.cuda.is_available() else -1,
        patience=20, save=True, save_period=10,
        project=str(CHECKPOINT_DIR), name='phase2_640px', exist_ok=True,
        lr0=0.0005, lrf=0.0001, warmup_epochs=5, warmup_momentum=0.8, momentum=0.937, weight_decay=0.0005,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=20, translate=0.15, scale=0.5,
        flipud=0.5, fliplr=0.5, mosaic=1.0, mixup=0.2, val=True, verbose=True
    )
    best = Path(r.save_dir) / 'weights' / 'best.pt'
    if best.exists():
        v = YOLO(str(best)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p2_map'] = float(v.box.map50)
        print(f'mAP@50: {hist["p2_map"]:.4f}')
    hist['p2_done'] = True
    save_hist()
else:
    print('Phase 2 done')


=== PHASE 2: 640px (70 epochs with focal loss tweaks) ===
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/checkpoints/detector_v2_retrain/dataset.yaml, degrees=20, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.0001, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/content/checkpoints/detector_v2_retrain/phase1_320px/weights/best.pt, momentum=0.

## Phase 3: High Resolution (1024px, 30 epochs, batch=16 fix)

In [15]:
RUN_P3 = True
if RUN_P3 and not hist['p3_done']:
    print('\n=== PHASE 3: 1024px (30 epochs, batch=16) ===')
    best = CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt'
    m = YOLO(str(best))
    r = m.train(
        data=str(yaml_path), imgsz=1024, epochs=30, batch=16,
        device=0 if torch.cuda.is_available() else -1,
        patience=15, save=True, save_period=5,
        project=str(CHECKPOINT_DIR), name='phase3_1024px', exist_ok=True,
        lr0=0.00025, lrf=0.00001, warmup_epochs=3, momentum=0.937, weight_decay=0.0005,
        hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, degrees=15, translate=0.1, scale=0.4,
        flipud=0.3, fliplr=0.5, mosaic=0.9, mixup=0.1, val=True, verbose=True
    )
    best = Path(r.save_dir) / 'weights' / 'best.pt'
    if best.exists():
        v = YOLO(str(best)).val(data=str(yaml_path), split='val', verbose=False)
        hist['p3_map'] = float(v.box.map50)
        print(f'mAP@50: {hist["p3_map"]:.4f}')
    hist['p3_done'] = True
    save_hist()
elif hist['p3_done']:
    print('Phase 3 done')
else:
    print('Phase 3 disabled')


=== PHASE 3: 1024px (30 epochs, batch=16) ===
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/checkpoints/detector_v2_retrain/dataset.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00025, lrf=1e-05, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/content/checkpoints/detector_v2_retrain/phase2_640px/weights/best.pt, momentum=0.937, mosaic=

## Best Model & Threshold Tuning

In [16]:
# Find best
best_phase = max(
    [(f'P1 (320px)', hist['p1_map'], CHECKPOINT_DIR / 'phase1_320px' / 'weights' / 'best.pt', hist['p1_done']),
     (f'P2 (640px)', hist['p2_map'], CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt', hist['p2_done']),
     (f'P3 (1024px)', hist['p3_map'], CHECKPOINT_DIR / 'phase3_1024px' / 'weights' / 'best.pt', hist['p3_done'])],
    key=lambda x: x[1] if x[3] else -1
)

name, score, path, _ = best_phase
print(f'Best: {name} (mAP@50={score:.4f})')

# Threshold sweep
print('\nTesting thresholds...')
m = YOLO(str(path))
best_conf = 0.5
best_map = 0
results = {}

for conf in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    m.conf = conf
    v = m.val(data=str(yaml_path), split='test', device=0 if torch.cuda.is_available() else -1, verbose=False)
    map50 = float(v.box.map50)
    results[conf] = {'map50': map50, 'precision': float(v.box.mp), 'recall': float(v.box.mr)}
    print(f'Conf {conf:.2f}: mAP@50={map50:.4f}')
    if map50 > best_map:
        best_map = map50
        best_conf = conf

print(f'\nOptimal: conf={best_conf:.2f} (mAP@50={best_map:.4f})')

# Save config
cfg = {
    'checkpoint': str(path),
    'confidence': float(best_conf),
    'map50': float(best_map),
    'precision': float(results[best_conf]['precision']),
    'recall': float(results[best_conf]['recall']),
    'thresholds': {float(k): v for k, v in results.items()},
    'model': 'YOLOv8l',
}
json.dump(cfg, open(CHECKPOINT_DIR / 'config.json', 'w'), indent=2)

print(f'\nConfig saved')

Best: P2 (640px) (mAP@50=0.7030)

Testing thresholds...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 113 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 85.5±110.2 MB/s, size: 1533.8 KB)
val: Scanning /content/checkpoints/detector_v2_retrain/test/labels... 97 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 97/97 239.3it/s 0.4s
val: New cache created: /content/checkpoints/detector_v2_retrain/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.1it/s 6.1s
                   all         97        174      0.546       0.69      0.552        0.3
Speed: 4.0ms preprocess, 48.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val-4
Conf 0.25: mAP@50=0.5522
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image

## ENSEMBLE Evaluation

In [17]:
# Ensemble Voting (Average Confidence across Phase 1, 2, 3)
print('\n=== ENSEMBLE EVALUATION ===')

p1_ckpt = CHECKPOINT_DIR / 'phase1_320px' / 'weights' / 'best.pt'
p2_ckpt = CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt'
p3_ckpt = CHECKPOINT_DIR / 'phase3_1024px' / 'weights' / 'best.pt'

test_img_dir = CHECKPOINT_DIR / 'test' / 'images'
test_images = sorted(list(test_img_dir.glob('*.*')))
print(f'Test images: {len(test_images)}')

# Collect predictions from all phases
from collections import defaultdict
all_preds = {}  # {img_stem: [conf_p1, conf_p2, conf_p3, ...]}

for ckpt_path in [p1_ckpt, p2_ckpt, p3_ckpt]:
    if not ckpt_path.exists():
        print(f'Skipping {ckpt_path.parent.name} (not found)')
        continue

    print(f'\nPredicting with {ckpt_path.parent.name}...')
    m = YOLO(str(ckpt_path))
    m.conf = best_conf

    for img_path in tqdm(test_images):
        r = m.predict(img_path, verbose=False)[0]
        img_stem = Path(img_path).stem
        if img_stem not in all_preds:
            all_preds[img_stem] = []

        if r.boxes is not None and len(r.boxes) > 0:
            confs = r.boxes.conf.cpu().numpy() if hasattr(r.boxes.conf, 'cpu') else r.boxes.conf
            all_preds[img_stem].append(float(confs.mean()) if len(confs) > 0 else 0.0)
        else:
            all_preds[img_stem].append(0.0)

# Average confidence across phases
ensemble_confs = {}
for img_stem, conf_list in all_preds.items():
    if conf_list:
        ensemble_confs[img_stem] = np.mean(conf_list)

avg_ensemble = np.mean(list(ensemble_confs.values())) if ensemble_confs else 0
print(f'\nEnsemble avg confidence: {avg_ensemble:.4f}')

# Validate each phase + ensemble on test split
print(f'\n--- Test Split Validation ---')
for phase, ckpt_path in [('P1 (320px)', p1_ckpt), ('P2 (640px)', p2_ckpt), ('P3 (1024px)', p3_ckpt)]:
    if ckpt_path.exists():
        m = YOLO(str(ckpt_path))
        v = m.val(data=str(yaml_path), split='test', device=0 if torch.cuda.is_available() else -1, verbose=False)
        print(f'{phase}: mAP@50 = {float(v.box.map50):.4f}')

print('\n✓ Ensemble evaluation complete')


=== ENSEMBLE EVALUATION ===
Test images: 97

Predicting with weights...


  0%|          | 0/97 [00:00<?, ?it/s]


Predicting with weights...


  0%|          | 0/97 [00:00<?, ?it/s]


Predicting with weights...


  0%|          | 0/97 [00:00<?, ?it/s]


Ensemble avg confidence: 0.1920

--- Test Split Validation ---
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 113 layers, 43,607,379 parameters, 0 gradients, 164.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2650.8±826.3 MB/s, size: 1477.5 KB)
val: Scanning /content/checkpoints/detector_v2_retrain/test/labels.cache... 97 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 97/97 33.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.0it/s 3.6s
                   all         97        174      0.495      0.598      0.494      0.241
Speed: 3.0ms preprocess, 15.2ms inference, 0.0ms loss, 4.0ms postprocess per image
Results saved to /content/runs/detect/val-14
P1 (320px): mAP@50 = 0.4939
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 113 layers, 43,607,379 parameters, 0 gradients, 164.8 G

## Save Best Model to Drive (if Colab)

In [18]:
if IN_COLAB:
    import shutil
    best_pt = CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt'
    drive_dir = DRIVE_DIR / 'checkpoints'
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_pt, drive_dir / 'detector_v2_retrain_best.pt')
    print(f'✓ Saved to Drive: {drive_dir / "detector_v2_retrain_best.pt"}')
else:
    best_pt = CHECKPOINT_DIR / 'phase2_640px' / 'weights' / 'best.pt'
    if best_pt.exists():
        print(f'✓ Best model: {best_pt}')
        print(f'Checkpoint dir: {CHECKPOINT_DIR}')
    else:
        print('Best model not found')

✓ Saved to Drive: /content/drive/MyDrive/HeritagePreservation/checkpoints/detector_v2_retrain_best.pt
